In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [5]:
import pandas as pd
from scipy.stats import boxcox
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "phi4-mini:3.8b" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

In [ ]:
def preprocess(df):
    X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
    y_a = df['a']

    X_train, X_temp = train_test_split(
        X, test_size=0.3, random_state=42
    )
    X_val, X_test = train_test_split(
        X_temp, test_size=0.5, random_state=42
    )

    y_a_train, y_a_val, y_a_test = y_a.loc[X_train.index], y_a.loc[X_val.index], y_a.loc[X_test.index]
    X_train_bc = X_train.copy()
    X_val_bc = X_val.copy()
    X_test_bc = X_test.copy()

    boxcox_params = {}  

    for col in X_train.columns:
        min_val = X_train[col].min()
        shift = 0.0
        if min_val <= 0:
            shift = -min_val + 1e-6

        X_train_shifted = X_train[col] + shift
        X_val_shifted = X_val[col] + shift
        X_test_shifted = X_test[col] + shift

        X_train_bc[col], lam = boxcox(X_train_shifted)

        X_val_bc[col] = boxcox(X_val_shifted, lam)
        X_test_bc[col] = boxcox(X_test_shifted, lam)
        
        boxcox_params[col] = {
            "lambda": lam,
            "shift": shift
        }
    return [X_train_bc, X_val_bc, X_test_bc, y_a_train, y_a_val, y_a_test]

In [ ]:
@tool
def run_preprocess(df) -> list:
    """
    Runs the preprocessing code and returns X train, X test, X valid and Y train, Y test, Y valid  data respectively.
    Pass the dataframe as the argument
    """
    return preprocess(df)


In [ ]:
def forecast(X_train_bc, X_val_bc,X_test_bc, y_a_train, y_a_val, y_a_test):
    xgb = XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=42
    )
    xgb.fit(X_train_bc, y_a_train)

    y_a_train_pred = xgb.predict(X_train_bc)
    y_a_val_pred = xgb.predict(X_val_bc)
    y_a_test_pred = xgb.predict(X_test_bc)
    return {
        "train_r2": r2_score(y_a_train, y_a_train_pred),
        "val_r2": r2_score(y_a_val, y_a_val_pred),
        "test_r2":  r2_score(y_a_test, y_a_test_pred),
        "test_mse": mean_squared_error(y_a_test, y_a_test_pred)
    }

In [ ]:
@tool
def run_forecast(df) -> dict:
    """
    Runs the forecasting pipeline and returns model results.
    Must be called before writing the report.   
    """
    return forecast(df)


### preprocess agent

In [ ]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert data analyst. "
        "perform EDA on the given dataset"
        ""
    ),
    (
        "human",
            """
            Here is data sample:
            {data}

            Perform EDA
            """
    ),
])


preprocess_chain = preprocess_prompt | llm


### report generation agent

In [ ]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert data analyst. "
        "You must base your analysis strictly on the provided evaluation results. "
    ),
    (
        "human",
            """
            Here are the forecasting results (JSON):
            {results}

            Write a markdown evaluation report including:
            - summary
            - Metrics table 
            - Interpretation
            """
    ),
])

report_chain = report_prompt | llm


In [ ]:
df = pd.read_csv("data/fitting-results.csv")
preprocessor = preprocess_chain.invoke({
    "data": df
})

markdown_report = report_chain.invoke({
    "results": tool_result
})

# print(markdown_report.content)
output_path = Path("result/forecast_report.md")
output_path.write_text(markdown_report.content, encoding="utf-8")


2062

### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 
